In [1]:
%load_ext autoreload
%autoreload 2

# !export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

In [2]:
import os
os.chdir('D:/thesis_code/keypoint_matcher')

from config import config
from utils import logger
import numpy as np
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# torch.cuda.empty_cache()

# Check Data Module

In [3]:
from src import MatchesDataModule
dm = MatchesDataModule()

dm.setup(stage='fit')
dl = dm.train_dataloader()

INFO     | setup | Train Dataset       : 137052 samples
INFO     | setup | Validation Dataset  : 50016 samples


In [4]:
batch = next(iter(dl))
ref_patches, references, tar_patches, targets, estimates, confidences = batch

print(f'{references.shape=}')

count = 24 * 2

# from utils import show_batch
# show_batch(
#     ref_patches, tar_patches,
#     references, estimates, targets,
#
#     confidences_true=confidences,
#     estimates=estimates,
#
#     limit_count=count,
#
#     n_columns=6,
#     just_gt=False,
# )

references.shape=torch.Size([12, 2])


# Light

In [5]:
# torch.cuda.empty_cache()

from src import Light
light = Light()

from utils import count_params
num = count_params(light.model)
print(f"{num:,}")

DEBUG    | __init__ | patch_size: 32, in_channels: 1, embedding_length: 32, out_channels: 512
4,820,970


In [6]:
target_coords = light(
    ref_patches, 
    tar_patches, 
    references,
    estimates,
)

print(f'{target_coords.shape=}')

DEBUG    | forward | 1 ref_patches.shape=torch.Size([12, 32, 32, 32])
DEBUG    | forward | 2 x.shape=torch.Size([12, 64, 32, 32])
DEBUG    | forward | 3 x.shape=torch.Size([12, 512, 16, 16])
DEBUG    | forward | 4 x.shape=torch.Size([12, 512, 1, 1])
DEBUG    | forward | 5 x.shape=torch.Size([12, 512])
DEBUG    | forward | 6 x.shape=torch.Size([12, 516])
DEBUG    | forward | 7 x.shape=torch.Size([12, 2])
DEBUG    | forward | 8 coords.shape=torch.Size([12, 2])
target_coords.shape=torch.Size([12, 2])
